In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
print("done")

done


In [20]:
# Load the dataset
df = pd.read_csv("creditcard.csv")

# Display class distribution before SMOTE
print("Class Distribution (before SMOTE):")
print(df["Class"].value_counts())


Class Distribution (before SMOTE):
Class
0    284315
1       492
Name: count, dtype: int64


In [21]:
# Separate features and target variable
X = df.drop(columns=["Class", "Time"])  # Drop 'Class' and 'Time'
y = df["Class"]

# Normalize the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("done")

done


In [22]:
# Handling Class Imbalance using SMOTE
smote = SMOTE(sampling_strategy=1.0, random_state=42)  # 1.0 means equal number of fraud and non-fraud cases
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

# Display new class distribution
print("\nNew Class Distribution (after SMOTE):")
print(pd.Series(y_resampled).value_counts())



New Class Distribution (after SMOTE):
Class
0    284315
1    284315
Name: count, dtype: int64


In [23]:
# Convert resampled data to DataFrame for easier viewing
X_resampled_df = pd.DataFrame(X_resampled)
y_resampled_df = pd.Series(y_resampled, name="Class")

# Concatenate features and target
resampled_df = pd.concat([X_resampled_df, y_resampled_df], axis=1)

# Display first 10 rows where Class == 1
fraud_rows = resampled_df[resampled_df["Class"] == 1].head(10)
print("First 10 rows with Class == 1 (Fraud):")
print(fraud_rows)


First 10 rows with Class == 1 (Fraud):
             0         1         2         3         4         5         6  \
541  -1.180495  1.182090 -1.061730  2.823647 -0.378330 -1.070764 -2.051091   
623  -1.553864 -1.912006  0.717864  1.616427  0.985192 -0.799255  0.263177   
4920 -1.175963  1.065368 -0.237259  1.645808 -0.595277 -0.056886  0.454550   
6108 -2.245363  0.822602 -1.710035  1.892684 -0.817341 -1.280925 -2.826143   
6329  0.630132  1.828699 -2.838971  3.342686  2.625768 -1.019123  1.385059   
6331  0.004304  2.505797 -4.115869  4.714946  0.556646 -2.516804 -1.319009   
6334  0.013672  2.502543 -4.326852  4.483868  0.963355 -1.886615 -1.365382   
6336  0.168273  2.248457 -3.809350  4.292966  1.208017 -1.816577 -0.657099   
6338  0.161566  2.306706 -3.703314  4.271198  1.125906 -1.990104 -0.603496   
6427  0.370475  1.393379 -3.515230  2.830552 -1.253699 -1.300183 -3.208003   

             7         8         9  ...        20        21        22  \
541   1.165200 -2.521403 -2.5

In [24]:
# Display first 10 rows where Class == 0
non_fraud_rows = resampled_df[resampled_df["Class"] == 0].head(10)
print("First 10 rows with Class == 0 (Non-Fraud):")
print(non_fraud_rows)


First 10 rows with Class == 0 (Non-Fraud):
          0         1         2         3         4         5         6  \
0 -0.694242 -0.044075  1.672773  0.973366 -0.245117  0.347068  0.193679   
1  0.608496  0.161176  0.109797  0.316523  0.043483 -0.061820 -0.063700   
2 -0.693500 -0.811578  1.169468  0.268231 -0.364572  1.351454  0.639776   
3 -0.493325 -0.112169  1.182516 -0.609727 -0.007469  0.936150  0.192071   
4 -0.591330  0.531541  1.021412  0.284655 -0.295015  0.071999  0.479302   
5 -0.217475  0.581675  0.752585 -0.118833  0.305009 -0.022313  0.384936   
6  0.627795  0.085389  0.029923  0.849383  0.139020  0.204695 -0.004170   
7 -0.328928  0.858692  0.708576 -0.347631  0.687512  0.321345  0.905860   
8 -0.456573  0.173291 -0.074653 -0.191774  1.934149  2.793594  0.299206   
9 -0.172698  0.678005  0.688781 -0.156927  0.361792 -0.185219  0.526706   

          7         8         9  ...        20        21        22        23  \
0  0.082637  0.331128  0.083386  ... -0.024923  0.3

In [25]:
# Step 3: Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, 
    test_size=0.2, random_state=42, stratify=y_resampled
)

# Confirm shape
print("\nTrain/Test Split:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


Train/Test Split:
X_train: (454904, 29)
X_test : (113726, 29)
y_train: (454904,)
y_test : (113726,)


In [26]:
from sklearn.feature_selection import SelectKBest, f_classif

# Select top k features (you can try with k=10, 15, etc.)
k = 5
selector = SelectKBest(score_func=f_classif, k=k)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Get feature names
selected_features = selector.get_support(indices=True)
print("Top features selected:", selected_features)


Top features selected: [ 3  9 10 11 13]


In [27]:
# Use only the selected features (top 5) for training
top_features = [3, 9, 10, 11, 13]  # Indices of top features

# Selecting features from X_train and X_test using numpy array indexing
X_train_selected = X_train[:, top_features]  # Selecting columns by index
X_test_selected = X_test[:, top_features]    # Selecting columns by index

print("done")


done


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Initialize Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)

# Train the model
lr_model.fit(X_train_selected, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test_selected)

# Evaluate the model
print("Logistic Regression Classification Report:\n", classification_report(y_test, y_pred_lr))


Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.98      0.94     56863
           1       0.97      0.90      0.94     56863

    accuracy                           0.94    113726
   macro avg       0.94      0.94      0.94    113726
weighted avg       0.94      0.94      0.94    113726



In [29]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# Initialize K-Nearest Neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model
knn_model.fit(X_train_selected, y_train)

# Make predictions
y_pred_knn = knn_model.predict(X_test_selected)

# Evaluate the model
print("KNN Classification Report:\n", classification_report(y_test, y_pred_knn))


KNN Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99     56863
           1       0.99      1.00      0.99     56863

    accuracy                           0.99    113726
   macro avg       0.99      0.99      0.99    113726
weighted avg       0.99      0.99      0.99    113726



In [30]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score

# Initialize XGBoost classifier (case-specific settings like 'eval_metric' included)
xgb_model = xgb.XGBClassifier(eval_metric='mlogloss')

# Train the model
xgb_model.fit(X_train_selected, y_train)

# Make predictions on test data
y_pred_xgb = xgb_model.predict(X_test_selected)

# Print classification report
print("XGBoost Classification Report:\n", classification_report(y_test, y_pred_xgb))

# Calculate and print accuracy with 4 decimal places
accuracy = accuracy_score(y_test, y_pred_xgb)
print(f"Accuracy: {accuracy:.4f}")



XGBoost Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99     56863
           1       0.99      0.99      0.99     56863

    accuracy                           0.99    113726
   macro avg       0.99      0.99      0.99    113726
weighted avg       0.99      0.99      0.99    113726

Accuracy: 0.9875


In [31]:
import sys
!{sys.executable} -m pip install lightgbm


In [32]:
import lightgbm as lgb
from sklearn.metrics import classification_report

# Initialize LightGBM classifier
lgb_model = lgb.LGBMClassifier()

# Train the model
lgb_model.fit(X_train_selected, y_train)

# Make predictions
y_pred_lgb = lgb_model.predict(X_test_selected)

# Evaluate the model
print("LightGBM Classification Report:\n", classification_report(y_test, y_pred_lgb))


[LightGBM] [Info] Number of positive: 227452, number of negative: 227452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008800 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 454904, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
LightGBM Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98     56863
           1       0.98      0.98      0.98     56863

    accuracy                           0.98    113726
   macro avg       0.98      0.98      0.98    113726
weighted avg       0.98      0.98      0.98    113726



C:\Users\Pallavi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [33]:
from sklearn.model_selection import cross_val_score
import lightgbm as lgb

# Reinitialize model
lgb_model = lgb.LGBMClassifier()

# Perform 5-fold cross-validation on the selected features
cv_scores = cross_val_score(lgb_model, X_train_selected, y_train, cv=5, scoring='f1')

print("Cross-Validation F1 Scores (5-fold):", cv_scores)
print("Mean F1 Score:", cv_scores.mean())


[LightGBM] [Info] Number of positive: 181962, number of negative: 181961
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005518 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 363923, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500001 -> initscore=0.000005
[LightGBM] [Info] Start training from score 0.000005


C:\Users\Pallavi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 181962, number of negative: 181961
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001224 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 363923, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500001 -> initscore=0.000005
[LightGBM] [Info] Start training from score 0.000005


C:\Users\Pallavi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 181961, number of negative: 181962
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001268 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 363923, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499999 -> initscore=-0.000005
[LightGBM] [Info] Start training from score -0.000005


C:\Users\Pallavi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 181961, number of negative: 181962
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001095 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 363923, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499999 -> initscore=-0.000005
[LightGBM] [Info] Start training from score -0.000005


C:\Users\Pallavi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 181962, number of negative: 181962
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001233 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 363924, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Cross-Validation F1 Scores (5-fold): [0.97843475 0.97923301 0.98064189 0.97987599 0.97903722]
Mean F1 Score: 0.9794445716868371


C:\Users\Pallavi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [34]:
# from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
import lightgbm as lgb
import numpy as np

# Step 2: Fit model to entire training set
lgb_model.fit(X_train_selected, y_train)

# Step 3: Evaluate on test set
y_pred = lgb_model.predict(X_test_selected)
print("Test Set Evaluation:\n")
print(classification_report(y_test, y_pred))

# Step 4: Predict on user input

# Updated feature names based on your top features
feature_names = ['feature_3', 'feature_9', 'feature_10', 'feature_11', 'feature_13']

#FIRST 3 POSITIVES , LAST 2 NEGATIVES.........
#TRY 0,0,0,0,0 FOR LEGIT CASE & 100,100,100,-100,-100 FOR FRAUDULENT........

print("\nEnter values for the following features to check if it's fraud:")

input_values = []
for feature in feature_names:
    val = float(input(f"{feature}: "))
    input_values.append(val)

input_array = np.array(input_values).reshape(1, -1)

# Step 5: Predict & Display
prediction = lgb_model.predict(input_array)

if prediction[0] == 1:
    print("\n⚠️ The transaction is predicted to be: FRAUDULENT.")
else:
    print("\n✅ The transaction is predicted to be: LEGITIMATE.")


[LightGBM] [Info] Number of positive: 227452, number of negative: 227452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001511 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 454904, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


C:\Users\Pallavi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Test Set Evaluation:

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     56863
           1       0.98      0.98      0.98     56863

    accuracy                           0.98    113726
   macro avg       0.98      0.98      0.98    113726
weighted avg       0.98      0.98      0.98    113726


Enter values for the following features to check if it's fraud:


feature_3:  0
feature_9:  0
feature_10:  0
feature_11:  0
feature_13:  0



✅ The transaction is predicted to be: LEGITIMATE.


C:\Users\Pallavi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [36]:
import joblib
import lightgbm as lgb

# Train your LightGBM model (assuming you've already done this)
# lgb_model = lgb.LGBMClassifier()  # Example model

# Save the model with joblib
joblib.dump(lgb_model, 'lgbm.keras')
print("done")

done
